# G03-A — Original Image-Level Ensemble Benchmark

**Kaggle notebook:** `g03a-image-level-all-ensembles-s42`  
**Required input:** completed `g02-a-v2-ham10000-bcn20000-source-evaluation` output, including `reports/g02a_status.json` and `predictions/base_*.npz`.  
**For exact historical CalStack-SSL replay:** also attach the original `confidence-gated-ensemble` notebook output containing `final_ensemble_config.json` and `stacking_meta_model.joblib`. Without these, historical trained stacking is visibly marked unavailable; no substitute is trained on test labels.  
**Compute:** CPU, Internet OFF; source probabilities already exist. **No dataset download** and no model training.  
**Statistical scope:** the 2,239-image *conservatively matched* original J1 test cohort from the uploaded G02-A run (counts may differ in another run), not fully unseen external HAM/BCN. The earlier matching excluded 295 unresolved/conflicting records. These are source-stratified results.  
**Class handling:** full eight-output predictions are primary; restricted output-space scores use the ground-truth class set of the evaluation cohort and are explicitly secondary/optimistic. Candidate parameters are frozen/prespecified, never optimized on this test subset.

## 1. Strict input loading and reusable evaluation functions

In [1]:
import os,json,hashlib,time,math
from pathlib import Path
from collections import OrderedDict
import numpy as np,pandas as pd,joblib
from scipy.special import softmax
from sklearn.metrics import f1_score,accuracy_score,log_loss

SEED=42
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path(os.getenv('G03_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').is_dir() else '/mnt/data')).resolve()
WORK=Path(os.getenv('G03_WORK_DIR','/kaggle/working' if Path('/kaggle/working').is_dir() else '/mnt/data')).resolve()

def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def jwrite(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));tmp.replace(path)
def get_json(p):return json.loads(Path(p).read_text())
def find_status(filename,stage):
    hits=[]
    for p in INPUT.rglob(filename):
        if not p.is_file():continue
        j=get_json(p)
        if j.get('stage')==stage and j.get('status')=='COMPLETED':hits.append((p,j))
    req(len(hits)==1,f'Expected one completed {stage} {filename}; found {len(hits)}. Attach its Kaggle notebook output. Paths: {[str(a[0]) for a in hits[:6]]}')
    return hits[0]
def normalized(a):
    a=np.asarray(a,dtype=np.float64)
    req(a.ndim==2 and a.shape[1]==8 and np.isfinite(a).all() and (a>=0).all(),'Invalid [N,8] predictions')
    a=np.maximum(a,1e-12)
    return a/a.sum(1,keepdims=True)
def tscale(p,t):
    req(np.isfinite(t) and float(t)>0,'Invalid temperature')
    return softmax(np.log(np.clip(p,1e-12,1))/float(t),axis=1)
def bias(p,b):
    b=np.asarray(b,dtype=float)
    req(b.shape==(8,) and np.isfinite(b).all(),'Invalid class-bias vector')
    return softmax(np.log(np.clip(p,1e-12,1))+b[None,:],axis=1)
def combine(d,keys,weights=None,kind='arithmetic'):
    P=np.stack([d[s] for s in keys],axis=0)
    w=np.ones(len(keys),dtype=float)/len(keys) if weights is None else np.asarray(weights,dtype=float)
    req(w.shape==(len(keys),) and np.isfinite(w).all() and (w>=0).all() and np.isclose(w.sum(),1,atol=2e-4),'Invalid ensemble weights')
    w=w/w.sum()
    if kind=='arithmetic':return normalized(np.einsum('snc,s->nc',P,w,optimize=True))
    if kind=='geometric':return softmax(np.einsum('snc,s->nc',np.log(np.clip(P,1e-12,1)),w,optimize=True),axis=1)
    if kind=='vote':
        onehot=np.eye(8)[P.argmax(2)]
        return normalized(np.einsum('snc,s->nc',onehot,w,optimize=True)+.0001*np.einsum('snc,s->nc',P,w,optimize=True))
    raise RuntimeError(f'Unknown combination: {kind}')
def gated(d,keys,measure,beta,prior=None):
    p=np.stack([d[s] for s in keys],axis=1)
    if measure=='confidence':q=p.max(2)
    elif measure=='entropy':q=1.+(p*np.log(np.clip(p,1e-12,1))).sum(2)/math.log(8)
    elif measure=='margin':
        top=np.partition(p,-2,axis=2)[:,:,-2:];q=top.max(2)-top.min(2)
    else:raise RuntimeError(f'Unknown gate measure {measure}')
    prior_term=0 if prior is None else np.log(np.asarray(prior,dtype=float)[None,:])
    w=softmax(float(beta)*q+prior_term,axis=1)
    return normalized(np.einsum('nsc,ns->nc',p,w,optimize=True))
def entropy(p):return -np.sum(p*np.log(np.clip(p,1e-12,1)),axis=1)
def stats_from_prob(p):
    t=np.partition(p,-2,axis=1)[:,-2:]
    return np.stack([p.max(1),t.max(1)-t.min(1),entropy(p)/math.log(8)],axis=1)
def stacked_features(d,keys,mode):
    x=np.concatenate([d[s] if mode=='raw' else np.log(np.clip(d[s],1e-12,1)) for s in keys],axis=1)
    if mode=='extended':x=np.concatenate([x]+[stats_from_prob(d[s]) for s in keys],axis=1)
    return x
def old_thesis_features(d,keys):
    parts=[]
    for s in keys:
        p=d[s];parts.extend([p,np.log(np.clip(p,1e-12,1)),p.max(1,keepdims=True),entropy(p)[:,None]])
    return np.concatenate(parts,axis=1).astype(np.float32)
def aligned_predict(model,x):
    clf=model.named_steps.get('clf',model.named_steps.get('classifier',None)) if hasattr(model,'named_steps') else model
    req(clf is not None and hasattr(clf,'classes_'),'Fitted stacking model missing classes_')
    cls=np.asarray(clf.classes_).astype(int)
    req(np.array_equal(cls,np.arange(8)),f'Unexpected trained stacker classes: {cls}')
    p=model.predict_proba(x)
    return normalized(p)
def ece(y,p,bins=15):
    pred=p.argmax(1);conf=p.max(1);edges=np.linspace(0,1,bins+1)
    out=0.
    for i in range(bins):
        mask=(conf>=edges[i]) & (conf<(edges[i+1] if i<bins-1 else 1.00000001))
        if mask.any():out+=mask.mean()*abs((pred[mask]==y[mask]).mean()-conf[mask].mean())
    return float(out)
def subset_score(y,p):
    y=np.asarray(y,dtype=int);p=normalized(p);pred=p.argmax(1)
    seen=np.unique(y)
    req(len(seen)>=2,'At least two represented diagnoses required')
    recalls=[(pred[y==c]==c).mean() for c in seen]
    missing=~np.isin(pred,seen)
    return dict(n=len(y),classes_present=';'.join(CLASSES[c] for c in seen),n_present=len(seen),
        balanced_accuracy=float(np.mean(recalls)),accuracy=float((pred==y).mean()),
        macro_f1=float(f1_score(y,pred,labels=seen,average='macro',zero_division=0)),
        macro_f1_eight=float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),
        nll_eight=float(log_loss(y,p,labels=np.arange(8))),ece_15=ece(y,p),
        unsupported_prediction_rate=float(missing.mean()),unsupported_predictions=int(missing.sum()))
def restricted(y,p):
    present=np.unique(np.asarray(y,dtype=int))
    a=np.asarray(p[:,present],dtype=float)
    z=np.zeros_like(p,dtype=float)
    z[:,present]=a/np.clip(a.sum(1,keepdims=True),1e-12,None)
    return z

def group_bootstrap(y,p,groups,B=300,seed=SEED):
    y=np.asarray(y,dtype=int);p=normalized(p);groups=np.asarray(groups,dtype=str)
    req(len(y)==len(p)==len(groups),'Bootstrap inputs not aligned')
    groups=np.asarray([f'unknown-image-{i}' if str(g).strip()=='' or str(g).upper().endswith(':UNKNOWN') else str(g) for i,g in enumerate(groups)])
    uniq,inv=np.unique(groups,return_inverse=True)
    indices=[np.flatnonzero(inv==i) for i in range(len(uniq))]
    rng=np.random.default_rng(seed);seen=set(y);values=[]
    for _ in range(B):
        take=np.concatenate([indices[i] for i in rng.integers(0,len(indices),len(indices))])
        yy=y[take]
        if set(yy)!=seen:continue
        pred=p[take].argmax(1)
        values.append(np.mean([(pred[yy==c]==c).mean() for c in seen]))
    return {'low':float(np.percentile(values,2.5)) if len(values)>=max(30,B//5) else None,
            'high':float(np.percentile(values,97.5)) if len(values)>=max(30,B//5) else None,
            'valid_resamples':len(values),'groups':len(uniq)}

def load_chunks(status_path,status,sources,stage):
    root=status_path.parent.parent
    chunks=sorted(status['chunks'],key=lambda x:x['start'])
    req(chunks and chunks[0]['start']==0 and chunks[-1]['end']==status['images'],'Incomplete chunk range')
    ids=[];ys=[];origin=[];group=[];data={s:[] for s in sources}
    offset=0
    for row in chunks:
        req(row['start']==offset and row['end']>offset,'Duplicate or missing chunk range')
        path=root/'predictions'/row['file']
        req(path.is_file() and sha(path)==row['sha256'],f'Prediction chunk missing or checksum mismatch: {path}')
        with np.load(path,allow_pickle=False) as z:
            n=row['end']-row['start']
            req('model_names' in z and list(z['model_names'].astype(str))==sources,'Source ordering mismatch')
            req('classes' in z and list(z['classes'].astype(str))==CLASSES,'Class ordering mismatch')
            req(len(z['image_id'])==n,'Chunk size mismatch')
            if stage=='G02-A':req(str(z['j1_split_sha256'].item())==status['j1_split_sha256'],'J1 split SHA mismatch')
            if stage=='G02-B':req(str(z['n01_manifest_sha256'].item())==status['n01_manifest_sha256'],'N01 manifest mismatch')
            ids.extend(z['image_id'].astype(str));ys.extend(z['y'].astype(int));origin.extend(z['source'].astype(str));group.extend(z['group_id'].astype(str))
            for s in sources:
                a=np.asarray(z['prob__'+s],dtype=float)
                req(a.shape==(n,8) and np.isfinite(a).all() and (a>=0).all() and np.max(abs(a.sum(1)-1))<2e-4,f'{s} bad source probabilities')
                data[s].append(a)
        offset=row['end']
    req(len(ids)==status['images']==status['processed'] and len(ids)==len(set(ids)),'Incomplete or duplicate image predictions')
    req(all(q in ['HAM10000','BCN20000'] for q in origin),'Unexpected source groups')
    req(all(0<=int(q)<8 for q in ys),'Unexpected diagnostic class index')
    return pd.DataFrame({'image_id':ids,'y':ys,'source':origin,'group_id':group}),{s:np.vstack(data[s]) for s in sources}

class Candidates:
    def __init__(self,n):self.n=n;self.p=OrderedDict();self.spec=OrderedDict()
    def add(self,name,family,a,spec):
        req(name not in self.p,f'Duplicate candidate: {name}')
        a=normalized(a);req(a.shape==(self.n,8),f'{name} shape mismatch')
        self.p[name]=a;self.spec[name]={'family':family,**spec}
    def evaluate(self,meta,out,track,scope,boot_names=()):
        out=Path(out);(out/'tables').mkdir(parents=True,exist_ok=True);(out/'predictions').mkdir(exist_ok=True);(out/'reports').mkdir(exist_ok=True)
        rows=[];recs=[];ci=[]
        y=meta.y.to_numpy(dtype=int);orig=meta.source.to_numpy(dtype=str)
        for name,p in self.p.items():
            for source in ['HAM10000','BCN20000']:
                idx=np.flatnonzero(orig==source);yy=y[idx];pp=p[idx]
                if len(idx)==0:continue
                base=subset_score(yy,pp)
                rows.append({'track':track,'scope':scope,'source':source,'method':name,'family':self.spec[name]['family'],'protocol':'full_eight_output',**base})
                narrow=restricted(yy,pp)
                rows.append({'track':track,'scope':scope,'source':source,'method':name,'family':self.spec[name]['family'],'protocol':'restricted_oracle_label_set',**subset_score(yy,narrow)})
                for c in np.unique(yy):
                    pred=pp.argmax(1);q=yy==c
                    recs.append({'source':source,'method':name,'class':CLASSES[c],'support':int(q.sum()),'recall':float((pred[q]==c).mean()),'protocol':'full_eight_output'})
                if name in boot_names:
                    b=group_bootstrap(yy,pp,meta.group_id.to_numpy(dtype=str)[idx],B=int(os.getenv('G03_BOOTSTRAP_N','300')),seed=SEED+int(source=='BCN20000'))
                    ci.append({'source':source,'method':name,**b})
        board=pd.DataFrame(rows).sort_values(['source','protocol','balanced_accuracy'],ascending=[True,True,False])
        board.to_csv(out/'tables/all_ensemble_source_metrics.csv',index=False)
        pd.DataFrame(recs).to_csv(out/'tables/per_class_recall_full_head.csv',index=False)
        pd.DataFrame(ci).to_csv(out/'tables/selected_bootstrap_95ci.csv',index=False)
        ckeys=list(self.p)
        ps=np.stack([self.p[k].astype(np.float32) for k in ckeys],axis=0)
        np.savez_compressed(out/'predictions/ensemble_probabilities.npz',image_id=meta.image_id.astype(str).to_numpy(),
            y=y.astype(np.int32),source=orig,group_id=meta.group_id.astype(str).to_numpy(),
            classes=np.asarray(CLASSES),candidate_names=np.asarray(ckeys),proba=ps)
        jwrite(out/'reports/all_candidate_specs.json',self.spec)
        print('Compared',len(self.p),'methods across HAM/BCN; evaluation rows',len(rows))
        print(board[(board.protocol=='full_eight_output')].groupby('source').head(12)[['source','method','family','balanced_accuracy','macro_f1','unsupported_prediction_rate']].round(4).to_string(index=False))
        return board


## 2. Verify completed model probabilities and cohort

In [2]:
OUT=WORK/'g03a-image-level-all-ensembles-s42'
(OUT/'reports').mkdir(parents=True,exist_ok=True)
SOURCES=['SimCLR_NoTTA','SimCLR_SelectedTTA','Rotation','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
status_path,status=find_status('g02a_status.json','G02-A')
req(status.get('models')==SOURCES,'G02-A model names do not match original thesis seven-source order')
meta,raw=load_chunks(status_path,status,SOURCES,'G02-A')
req(meta.source.value_counts().to_dict().get('HAM10000',0)>0 and meta.source.value_counts().to_dict().get('BCN20000',0)>0,'G02-A source populations incomplete')
req(meta.image_id.is_unique,'Duplicate official IDs')
cohort_path=status_path.parent/'g02a_verified_test_cohort.csv'
if cohort_path.is_file():
    cohort=pd.read_csv(cohort_path,dtype=str,keep_default_na=False)
    req(set(meta.image_id)==set(cohort.image_id),'G02-A cohort/predictions IDs disagree')
    ref=cohort.set_index('image_id')
    req(np.array_equal(meta.y.to_numpy(),ref.loc[meta.image_id,'label_id'].astype(int).to_numpy()),'Ground truth and probabilities disagree')
    req(np.array_equal(meta.source.to_numpy(),ref.loc[meta.image_id,'source'].to_numpy()),'Source provenance mismatch')
print('G03-A loaded:',len(meta),'source mix:',meta.source.value_counts().to_dict(),'J1_SHA:',status['j1_split_sha256'])



G03-A loaded: 2239 source mix: {np.str_('BCN20000'): 1240, np.str_('HAM10000'): 999} J1_SHA: 0fee67d07f343b31246a11c34b08e81d8d5645e0f260088aabf923fc91ac02eb


## 3. Evaluate predefined ensemble families without fitting

In [3]:
# No fitting, temperature-search, stacking-search, or class-bias optimization is allowed here.
reg=Candidates(len(meta))
for s in SOURCES:reg.add('Base_'+s,'Single model',raw[s],{'kind':'base','source':s})

# Predeclared reference and ablation families. These presets are NOT falsely labelled thesis-optimized.
subsets={'SSL_Rotation_SimCLR':['Rotation','SimCLR_NoTTA'],
         'SSL_WithTTA':['Rotation','SimCLR_NoTTA','SimCLR_SelectedTTA'],
         'SimCLR_Residual':['SimCLR_NoTTA','ResidualFusion'],
         'Four_Fusions':SOURCES[3:],'All7':SOURCES}
for name,keys in subsets.items():
    for kind in ['arithmetic','geometric','vote']:
        reg.add(f'{kind}_{name}',f'Fixed {kind}',combine(raw,keys,kind=kind),
                {'kind':kind,'sources':keys,'weights':[1/len(keys)]*len(keys),'fit':'PREDECLARED'})
for pair in [['SimCLR_NoTTA','ResidualFusion'],['Rotation','SimCLR_NoTTA'],['SimCLR_NoTTA','FusionConcat']]:
    for w in [.2,.4,.6,.8]:
        name=f'Pair_{pair[0]}_{pair[1]}_a{w:g}'
        reg.add(name,'Fixed pairwise weighting',combine(raw,pair,[w,1-w]),
                {'kind':'weighted','sources':pair,'weights':[w,1-w],'fit':'PREDECLARED'})
for anchor in ['SimCLR_NoTTA','ResidualFusion']:
    for a in [.2,.4,.6,.8]:
        w=np.full(len(SOURCES),(1-a)/(len(SOURCES)-1))
        w[SOURCES.index(anchor)]=a
        reg.add(f'AnchorWeighted_{anchor}_a{a:g}','Fixed anchor weighting',combine(raw,SOURCES,w),
                {'kind':'weighted','sources':SOURCES,'weights':w.tolist(),'fit':'PREDECLARED'})
for name,keys in {'SSL':subsets['SSL_Rotation_SimCLR'],'SimCLRResidual':subsets['SimCLR_Residual'],'All7':SOURCES}.items():
    for metric in ['confidence','entropy','margin']:
        for beta in [0.,1.,2.,4.,8.]:
            reg.add(f'Gate_{metric}_{name}_beta{beta:g}','Fixed adaptive gate',gated(raw,keys,metric,beta),
                    {'kind':'gate','sources':keys,'measure':metric,'beta':beta,'fit':'PREDECLARED'})
for name,anchor in [('SimCLR','SimCLR_NoTTA'),('Residual','ResidualFusion')]:
    avg=reg.p['arithmetic_All7']
    for a in [.2,.4,.6,.8]:
        reg.add(f'Blend_{name}_MeanAll7_alpha{a:g}','Fixed anchor blend',
                normalized((1-a)*raw[anchor]+a*avg),
                {'kind':'blend','anchor':anchor,'other':'arithmetic_All7','alpha':a,'fit':'PREDECLARED'})



## 4. Recover and replay original fitted ensembles

In [4]:
# Historical calibrated/optimized ensemble replay. No approximating missing fitted estimators.
legacy=[]
for p in INPUT.rglob('final_ensemble_config.json'):
    try:
        c=get_json(p)
        if c.get('models')==SOURCES and {'optimized_weights','stacking_alpha','class_bias','temperatures'}.issubset(c):
            legacy.append((p,c))
    except (ValueError,OSError,KeyError,TypeError):continue
req(len(legacy)<=1,f'Multiple historical thesis configs attached: {[str(p) for p,_ in legacy]}')
historical={'available':False,'reason':'Original thesis fitted config not mounted; no historical CalStack claim',
            'required':'original confidence-gated-ensemble output: final_ensemble_config.json and stacking_meta_model.joblib'}
if legacy:
    config_path,cfg=legacy[0]
    req(set(cfg['temperatures'])==set(SOURCES),'Historical calibration source set mismatch')
    req(set(cfg['optimized_weights'])==set(SOURCES),'Historical weighting source set mismatch')
    cal={s:tscale(raw[s],float(cfg['temperatures'][s])) for s in SOURCES}
    for s in SOURCES:
        reg.add(f'OriginalCal_{s}','Original calibrated single',cal[s],{'kind':'calibrated_base','source':s,'temperature':float(cfg['temperatures'][s]),'fit':'ORIGINAL_THESIS_VALIDATION'})
    w=np.array([cfg['optimized_weights'][s] for s in SOURCES],dtype=float)
    req((w>=0).all() and np.isclose(w.sum(),1,atol=2e-4),'Original constrained weights invalid')
    weighted=combine(cal,SOURCES,w)
    reg.add('OriginalThesis_ConstrainedWeighted','Original thesis trained weighting',weighted,
            {'kind':'weighted','sources':SOURCES,'weights':w.tolist(),'calibrated':True,'fit':'ORIGINAL_THESIS_VALIDATION'})
    for sub,keys in subsets.items():
        for kind in ['arithmetic','geometric','vote']:
            reg.add(f'OriginalCal_{kind}_{sub}','Original-calibrated reference',combine(cal,keys,kind=kind),
                    {'kind':kind,'sources':keys,'calibrated':True,'fit':'ORIGINAL_THESIS_TEMPERATURES'})
    stack_file=config_path.parent/'stacking_meta_model.joblib'
    if stack_file.is_file():
        old_model=joblib.load(stack_file)
        x=old_thesis_features(cal,SOURCES)
        stack=aligned_predict(old_model,x)
        reg.add('OriginalThesis_StackOnly','Original thesis trained stacker',stack,{'kind':'historical_stacker','fit':'ORIGINAL_THESIS_VALIDATION'})
        alpha=float(cfg['stacking_alpha']);req(0<=alpha<=1,'Historical blending alpha invalid')
        blended=normalized((1-alpha)*cal['SimCLR_NoTTA']+alpha*stack)
        reg.add('OriginalThesis_StackBlend_NoBias','Original CalStack ablation',blended,
                {'kind':'historical_blend','alpha':alpha,'anchor':'SimCLR_NoTTA','fit':'ORIGINAL_THESIS_VALIDATION'})
        b=cfg['class_bias']
        reg.add('OriginalThesis_CalStack_ClassBias','Original thesis proposed CalStack-SSL',bias(blended,b),
                {'kind':'historical_class_bias','alpha':alpha,'bias':b,'fit':'ORIGINAL_THESIS_VALIDATION'})
        historical={'available':True,'config_path':str(config_path),'config_sha256':sha(config_path),
                    'stacker_path':str(stack_file),'stacker_sha256':sha(stack_file),'final_method':cfg.get('final_method')}
    else:
        historical={'available':False,'partial':True,'config_path':str(config_path),'reason':'Original weighting replayed, but fitted stacking_meta_model.joblib is missing'}



## 5. Optional sources and robustness checks

In [5]:
# Optional supervised predictions are never used to refit the seven-source historical stacker.
sup_files=sorted((status_path.parent.parent/'predictions').glob('supervised_*.npz'))
if sup_files:
    sup={}
    for f in sup_files:
        with np.load(f,allow_pickle=False) as z:
            req('image_id' in z and 'proba' in z,'Malformed optional supervised prediction file')
            for img,p in zip(z['image_id'].astype(str),z['proba']):
                req(img not in sup,'Duplicate supervised image ID')
                sup[img]=p
    if set(sup)==set(meta.image_id):
        reg.add('Base_Supervised_ResNet50_5seed','Optional supervised baseline',np.stack([sup[i] for i in meta.image_id]),
                {'kind':'optional_base','fit':'FROZEN_PRETRAINED'})
    else:print('Optional supervised predictions incomplete; excluded from equal-cohort comparison')



## 6. Results, frozen prediction export and status

In [6]:
bootstrap=['Base_SimCLR_NoTTA','Base_ResidualFusion','arithmetic_All7','geometric_All7','vote_All7',
           'OriginalThesis_ConstrainedWeighted','OriginalThesis_StackBlend_NoBias','OriginalThesis_CalStack_ClassBias']
board=reg.evaluate(meta,OUT,'image_aware','G02-A_original_test_source_stratified',boot_names=set(bootstrap)&set(reg.p))
jwrite(OUT/'reports/historical_replay_availability.json',historical)
jwrite(OUT/'reports/final_status.json',{'stage':'G03-A','status':'COMPLETED','candidate_count':len(reg.p),
    'n_images':len(meta),'g02a_status_sha256':sha(status_path),'j1_split_sha256':status['j1_split_sha256'],
    'historical_replay':historical,'source_images':meta.source.value_counts().to_dict(),
    'selection':'NONE: no candidate chosen on evaluation cohort',
    'scope':'Original image-level test source-stratified, conservative image crosswalk; not independent HAM/BCN external generalization',
    'source_cohort_incomplete':True,'no_test_fitting':True})
print('G03-A done:',len(reg.p),'methods; original historical stack replay',historical['available'])


Compared 121 methods across HAM/BCN; evaluation rows 484
  source                               method                                family  balanced_accuracy  macro_f1  unsupported_prediction_rate
BCN20000     OriginalThesis_StackBlend_NoBias            Original CalStack ablation             0.8268    0.7992                          0.0
BCN20000               arithmetic_SSL_WithTTA                      Fixed arithmetic             0.8262    0.8348                          0.0
BCN20000    OriginalThesis_CalStack_ClassBias Original thesis proposed CalStack-SSL             0.8250    0.7955                          0.0
BCN20000      Pair_Rotation_SimCLR_NoTTA_a0.2              Fixed pairwise weighting             0.8216    0.8277                          0.0
BCN20000             OriginalThesis_StackOnly       Original thesis trained stacker             0.8177    0.7711                          0.0
BCN20000                geometric_SSL_WithTTA                       Fixed geometric        